# Unihra SDK: Interactive SEO Analysis

This notebook demonstrates how to use the `unihra` Python client for professional SEO analysis directly in Jupyter.

**Features covered:**
1. Setup and Authentication
2. Running analysis with a visual Progress Bar
3. Working with data using Pandas
4. Exporting reports to Excel

In [ ]:
# Install dependencies (if not already installed)
!pip install unihra pandas openpyxl tqdm matplotlib

In [ ]:
import os
import pandas as pd
from unihra import UnihraClient

# Best practice: Load key from environment or paste it securely
API_KEY = "YOUR_API_KEY_HERE"

# Initialize Client with Smart Retries enabled
client = UnihraClient(api_key=API_KEY, max_retries=3)

## 1. Run Analysis
We use `verbose=True` to enable the interactive progress bar.

In [ ]:
own_page = "https://example.com/product-page"
competitors = [
    "https://competitor.com/similar-item",
    "https://another-shop.com/best-seller"
]
# Critical for Semantic Gap analysis
target_queries = ["seller"]
# Run synchronous analysis
try:
    result = client.analyze(
        own_page=own_page, 
        competitors=competitors,
        queries=target_queries,
        lang="ru",
        verbose=True  # <--- Triggers the visual bar
    )
    print("Analysis completed successfully!")
except Exception as e:
    print(f"Error: {e}")

## 2. Data Exploration (Pandas)
Convert the JSON result into a readable DataFrame.

In [ ]:
# Extract the Semantic Context Analysis data
gaps_data = result.get('semantic_context_analysis', [])
df_gaps = pd.DataFrame(gaps_data)

# Show top 10 actionable recommendations
# Observe 'context_snippet' and 'coverage_percent' fields
if not df_gaps.empty:
    cols = ['lemma', 'recommendation', 'context_snippet', 'coverage_percent', 'gap']
    # Filter only available columns to prevent errors if data is missing
    available_cols = [c for c in cols if c in df_gaps.columns]
    
    display(df_gaps[available_cols].head(10))
else:
    print("No semantic gaps found.")

## 3. Visualization
Quickly visualize the difference in word frequency.

In [ ]:
# Convert the 'block_comparison' section to a DataFrame
df = client.get_dataframe(result, section="block_comparison")

# Show top 10 words where action is needed
df_action = df[df['action_needed'] != 'ok']
df_action[['word', 'frequency', 'action_needed']].head(10)

## 4. Structural Comparison Table
Extracts technical metrics (Title, H1, Text Volume, Uniqueness) to compare the Target Page directly against Competitors side-by-side.

In [ ]:
structures = result.get('page_structure', [])

if structures:
    flat_data = []
    for page in structures:
        is_target = page['url'] == own_page
        marker = "🟢 Target" if is_target else "🔴 Competitor"
        
        flat_data.append({
            "Type": marker,
            "URL": page['url'],
            "Meta Title": page['meta_tags'].get('title', '')[:60] + "...", # Truncate for display
            "Title Len": page['meta_tags'].get('title_length'),
            "H1 Header": page['content'].get('h1_heading', ''),
            "Text Volume (chars)": page['metrics'].get('char_count_no_spaces'),
            "Uniqueness (%)": page['metrics'].get('uniqueness_percentage')
        })

    df_struct = pd.DataFrame(flat_data)
    
    # Adjust pandas display options for better readability
    pd.set_option('display.max_colwidth', None)
    
    print("📊 Page Structure Comparison:")
    display(df_struct)
else:
    print("⚠️ No structure data available.")

## 5. Heading Hierarchy Drill-Down
Provides a deep dive into the document outline (H1-H6) for every analyzed page to understand the content structure strategy.

In [ ]:
if structures:
    print("📑 Document Outline (H1-H6 Hierarchy):\n")
    
    for page in structures:
        label = "TARGET PAGE" if page['url'] == own_page else "COMPETITOR"
        print(f"--- {label} --- {page['url']}")
        
        raw_headers = page['content'].get('heading_structure_raw', '')
        
        if raw_headers:
            headers_list = raw_headers.split('; ')
            # Display first 10 headers to keep output clean in the notebook
            for h in headers_list[:10]: 
                print(f"  └ {h}")
            if len(headers_list) > 10:
                print(f"  └ ... and {len(headers_list)-10} more headers")
        else:
            print("  └ No headers found.")
        print("\n")

## 6. Export Report
Save the full data to Excel for the marketing team.

In [ ]:
client.save_report(result, "seo_analysis_report.xlsx", style_output=True)
print("Saved to seo_analysis_report.xlsx")